# 3.3 Conditional Probability

**Goal:** reason about probabilities after information is observed and choose the right factorization for joint events.

**Prerequisites:** [Probability Rules](2_Probability-Rules.ipynb), set operations, and fractions. **Runtime:** Python 3.10+, standard library only. Synthetic examples have saved outputs. Run cells in order.

## Contents
1. Conditional probability
2. Independent events
3. Dependent events
4. Chain rule of probability
5. Law of total probability
6. Conditional independence and the Naive Bayes connection
7. Practice and solutions

## 1. Conditional Probability

For $P(B)>0$,
$$P(A\mid B)=\frac{P(A\cap B)}{P(B)}.$$
Conditioning restricts attention to B and renormalizes its probability to 1. It is not the same as a joint probability, and in general $P(A\mid B)\ne P(B\mid A)$.

**Worked example:** among 100 messages, 20 are spam. Of the spam messages, 15 contain the word “offer”; among the 80 nonspam messages, 10 contain it. Then
$$P(\text{spam}\mid\text{offer})=15/25=0.6,$$
while $P(\text{offer}\mid\text{spam})=15/20=0.75$ and $P(\text{spam}\cap\text{offer})=15/100=0.15$.

These are empirical proportions in a defined dataset. Using them for future messages requires a relevant sample, reliable labels, and a stable enough data-generating process.

If $P(B)=0$, the elementary ratio is undefined. Later probability theory handles some zero-probability conditioning through densities and conditional distributions; do not divide by zero or assign a value by convenience here.

In [1]:
from fractions import Fraction
# Joint-count table: class by word presence.
counts = {("spam", True): 15, ("spam", False): 5,
          ("not_spam", True): 10, ("not_spam", False): 70}
print("P(spam | offer):", Fraction(15, 15+10))
print("P(offer | spam):", Fraction(15, 15+5))
print("P(spam and offer):", Fraction(15, sum(counts.values())))
assert Fraction(15, 25) != Fraction(15, 20)

P(spam | offer): 3/5
P(offer | spam): 3/4
P(spam and offer): 3/20


## 2. Independent Events

Events A and B are independent when
$$P(A\cap B)=P(A)P(B).$$
This definition also applies to zero-probability events. When the conditioning probability is positive, independence is equivalent to $P(A\mid B)=P(A)$ or $P(B\mid A)=P(B)$. Observing one event does not change the other's probability under the model.

**Worked example:** in two independent fair coin tosses, let A mean first toss is heads and B mean second is heads. Both marginals are 1/2 and their intersection is HH with probability 1/4.

Independence is an assumption or property of the joint model, not a consequence of events occurring at different times or being recorded in different columns. Two purchases by the same user may be dependent even on different days.

**Pairwise versus mutual independence:** three events can be pairwise independent without being mutually independent. For two fair tosses, define C as “both tosses agree.” A, B, and C are independent in pairs, but $P(A\cap B\cap C)=1/4\ne1/8=P(A)P(B)P(C)$. Full mutual independence requires the product relation for every subcollection, not only pairs.

In [2]:
from itertools import product
omega = set(product("HT", repeat=2))
A = {outcome for outcome in omega if outcome[0] == "H"}
B = {outcome for outcome in omega if outcome[1] == "H"}
C = {outcome for outcome in omega if outcome[0] == outcome[1]}
def p(event):
    return Fraction(len(event), len(omega))
for name, first, second in [("AB", A, B), ("AC", A, C), ("BC", B, C)]:
    print(name, "intersection:", p(first & second), "product:", p(first)*p(second))
    assert p(first & second) == p(first)*p(second)
print("Triple intersection:", p(A & B & C), "triple product:", p(A)*p(B)*p(C))
assert p(A & B & C) != p(A)*p(B)*p(C)

AB intersection: 1/4 product: 1/4
AC intersection: 1/4 product: 1/4
BC intersection: 1/4 product: 1/4
Triple intersection: 1/4 triple product: 1/8


## 3. Dependent Events

Events are dependent when the product condition fails. In draws without replacement, one observed draw changes the remaining composition, so the next draw's probability can change.

**Worked example:** a bag has 3 red and 2 blue balls. Before drawing, the second draw's marginal probability of red is 3/5 by symmetry. Given the first draw was red, it is 2/4; given blue, it is 3/4. Hence first-red and second-red are dependent.

Mutually exclusive events with positive probabilities are dependent because one makes the other impossible. Dependence does not by itself establish a causal relationship: a common cause, selection process, or deterministic relationship can explain it.

## 4. Chain Rule of Probability

Repeated application of the multiplication rule gives
$$P(A\cap B\cap C)=P(A)P(B\mid A)P(C\mid A\cap B),$$
when these conditionals are defined. More generally,
$$P(A_1\cap\cdots\cap A_n)=P(A_1)\prod_{i=2}^nP(A_i\mid A_1\cap\cdots\cap A_{i-1}).$$
This does not assume independence. Factor ordering can be changed, but the conditioning histories must change consistently. Under mutual independence, all the conditional factors reduce to the corresponding marginals.

**Worked example:** three ordered draws without replacement from 3 red and 2 blue balls have probability of red, red, blue equal to
$$(3/5)(2/4)(2/3)=1/5.$$
This is for the specified order. “Exactly two red in any order” additionally includes RBR and BRR, giving 3/5 here. Do not multiply by a number of orders without checking whether those orders have the same probability.

In sequence modeling, $P(x_1,\ldots,x_n)=\prod_iP(x_i\mid x_1,\ldots,x_{i-1})$ is a chain-rule factorization of a joint distribution. It is not an independence assertion.

In [3]:
from itertools import permutations
balls = ["R1", "R2", "R3", "B1", "B2"]
draws = list(permutations(balls, 3))
rrb = [d for d in draws if tuple(ball[0] for ball in d) == ("R", "R", "B")]
exactly_two = [d for d in draws if sum(ball.startswith("R") for ball in d) == 2]
chain_probability = Fraction(3, 5)*Fraction(2, 4)*Fraction(2, 3)
print("Ordered RRB:", chain_probability)
print("Exactly two red, any order:", Fraction(len(exactly_two), len(draws)))
assert chain_probability == Fraction(len(rrb), len(draws)) == Fraction(1, 5)

Ordered RRB: 1/5
Exactly two red, any order: 3/5


## 5. Law of Total Probability

Let $B_1,\ldots,B_k$ be a partition of the sample space, with each conditioning cell having positive probability. Then
$$P(A)=\sum_{j=1}^kP(A\mid B_j)P(B_j).$$
The events $A\cap B_j$ are disjoint and together cover A, so the formula follows from addition and multiplication. It is a weighted average of conditional probabilities, with partition probabilities as weights. Zero-probability cells can be omitted rather than assigned undefined conditionals.

**Worked example:** 60% of requests come from mobile and 40% from web. Failure rates are 5% and 2%, respectively. Overall failure probability is
$$0.60(0.05)+0.40(0.02)=0.038.$$
An unweighted average of 5% and 2% would answer a different question. If the traffic mix changes, the aggregate rate can change without any subgroup rate changing.

Exhaustive overlapping categories do not form a partition and cannot be inserted into this sum without handling overlap. All component probabilities must refer to a compatible population and time window.

In [4]:
mobile_share, web_share = Fraction(3, 5), Fraction(2, 5)
fail_mobile, fail_web = Fraction(5, 100), Fraction(2, 100)
overall = mobile_share*fail_mobile + web_share*fail_web
new_mix = Fraction(9, 10)*fail_mobile + Fraction(1, 10)*fail_web
print("Original aggregate failure probability:", float(overall))
print("New 90% mobile mix, unchanged subgroup rates:", float(new_mix))
# Total probability also recovers the second-draw red marginal.
second_red = Fraction(3, 5)*Fraction(2, 4)+Fraction(2, 5)*Fraction(3, 4)
print("Marginal P(second red):", second_red)
assert second_red == Fraction(3, 5)

Original aggregate failure probability: 0.038
New 90% mobile mix, unchanged subgroup rates: 0.047
Marginal P(second red): 3/5


## 6. Conditional independence and AI/ML

A and B are conditionally independent given class C when, within each relevant class value $c$,
$$P(A\cap B\mid C=c)=P(A\mid C=c)P(B\mid C=c).$$
Conditional independence is different from marginal independence. Mixing class-specific distributions can create a marginal association even if the features are independent within each class. Conversely, marginal independence does not guarantee independence after conditioning.

**Naive Bayes** assumes features are conditionally independent given the class to simplify $P(x_1,\ldots,x_d\mid C)$ into $\prod_jP(x_j\mid C)$. It does not require that the features be independent overall. This modeling approximation can work for prediction even when imperfect, but correlated evidence may make probabilities too confident.

**Worked example:** two feature indicators each have probability 0.8 in class 1 and 0.2 in class 0, and classes are equally likely. Assume independence within each class. Each feature's marginal is 0.5, but their joint marginal is $0.5(0.8^2)+0.5(0.2^2)=0.34$, not 0.25.

In [5]:
p_class1 = Fraction(1, 2)
p_feature1 = p_class1*Fraction(4, 5)+(1-p_class1)*Fraction(1, 5)
p_feature2 = p_feature1
p_both = p_class1*Fraction(4, 5)**2+(1-p_class1)*Fraction(1, 5)**2
print("Feature marginals:", p_feature1, p_feature2)
print("Joint marginal:", p_both, "product of marginals:", p_feature1*p_feature2)
assert p_both == Fraction(17, 50)
assert p_both != p_feature1*p_feature2

Feature marginals: 1/2 1/2
Joint marginal: 17/50 product of marginals: 1/4


## Practice

1. A table contains 12 spam messages with “offer,” 8 spam without it, 18 nonspam with it, and 62 nonspam without it. Find $P(\text{spam}\mid\text{offer})$ and $P(\text{offer}\mid\text{spam})$.
2. If $P(A)=0.4$, $P(B)=0.5$, and $P(A\cap B)=0.2$, are A and B independent?
3. Draw red then blue without replacement from 3 red and 2 blue balls. Find the probability.
4. In a sequential process, $P(A)=0.6$, $P(B\mid A)=0.5$, and $P(C\mid A\cap B)=0.2$. Find the triple intersection.
5. 70% of items are made by factory A (defect rate 1%), and 30% by B (defect rate 4%). Find the aggregate defect probability.
6. Does pairwise independence of three events imply mutual independence?
7. Does Naive Bayes require features to be marginally independent?
8. Can one use the elementary conditional formula when the conditioning event has probability zero?

## Solutions

1. $12/(12+18)=0.4$; $12/(12+8)=0.6$. The denominators differ.
2. Yes, for these two events, because $0.2=0.4(0.5)$.
3. $(3/5)(2/4)=3/10$. The blue count stays 2 after the red draw, while the total becomes 4.
4. $0.6(0.5)(0.2)=0.06$; no independence assumption is required for these conditional factors.
5. $0.7(0.01)+0.3(0.04)=0.019$, or 1.9%.
6. No. The first-head, second-head, and agreement events above are a counterexample.
7. No. Its assumption is conditional independence given the class, not independence in the mixed population.
8. No. The ratio has a zero denominator; more advanced conditional-distribution machinery is needed in continuous settings.

**Readiness check:** choose the conditioning population carefully, retain complete histories in the chain rule, and verify that total-probability weights form a partition.

**Next:** [3.4 Bayes' Theorem](4_Bayes-Theorem.ipynb).